# Qasper exploratory data analysis

This notebook downloads `allenai/qasper`, flattens its nested records, checks the data, and saves the tables and figures used during model preparation.

The summary written to `eda.json` is used to choose preprocessing rules and context limits for the fine-tuning notebooks.

## 1. Setup

Import the required libraries, set the random seed, configure the plots, and record package versions.

In [1]:
import importlib
import json
import random
import re
import shutil
import statistics
import sys
import unicodedata
import hashlib
from collections import Counter
from datetime import datetime, timezone
from functools import lru_cache
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any, Dict, List


# Bootstrap only the packages needed for this notebook.
def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        package = pip_name or import_name
        print(f"Installing missing package: {package}")
        __import__("subprocess").check_call([sys.executable, "-m", "pip", "install", package, "-q"])


for module_name, pip_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("datasets", "datasets"),
    ("transformers", "transformers"),
    ("pyarrow", "pyarrow"),
]:
    ensure_package(module_name, pip_name)

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import DatasetDict, load_dataset
from transformers import AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

PACKAGE_VERSIONS = {}
for pkg in ["numpy", "pandas", "matplotlib", "seaborn", "datasets", "transformers", "pyarrow"]:
    try:
        PACKAGE_VERSIONS[pkg] = version(pkg)
    except PackageNotFoundError:
        PACKAGE_VERSIONS[pkg] = "not-found"

print("Package versions:")
for k, v in PACKAGE_VERSIONS.items():
    print(f"- {k}: {v}")

Package versions:
- numpy: 2.4.2
- pandas: 2.3.3
- matplotlib: 3.10.6
- seaborn: 0.13.0
- datasets: 3.6.0
- transformers: 4.43.4
- pyarrow: 22.0.0


## 2. Paths

Define the folders used for raw data, CSV files, processed data, and figures.

In [2]:
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (candidate / "data").exists():
        PROJECT_ROOT = candidate
        break

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CSV_DIR = DATA_DIR / "csv"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
HF_CACHE_DIR = DATA_DIR / "hf_cache"
HF_LOCAL_DATASET_DIR = RAW_DIR / "qasper_hf_dataset"
EDA_JSON_PATH = DATA_DIR / "eda.json"

for path in [DATA_DIR, RAW_DIR, CSV_DIR, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, HF_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

OUTPUT_PATHS = {
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "raw_dir": str(RAW_DIR),
    "csv_dir": str(CSV_DIR),
    "processed_dir": str(PROCESSED_DIR),
    "reports_dir": str(REPORTS_DIR),
    "figures_dir": str(FIGURES_DIR),
    "hf_dataset_dir": str(HF_LOCAL_DATASET_DIR),
    "eda_json": str(EDA_JSON_PATH),
}

OUTPUT_PATHS

{'project_root': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning',
 'data_dir': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\data',
 'raw_dir': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\data\\raw',
 'csv_dir': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\data\\csv',
 'processed_dir': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\data\\processed',
 'reports_dir': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\reports',
 'figures_dir': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\reports\\figures',
 'hf_dataset_dir': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\data\\raw\\qasper_hf_dataset',
 'eda_json': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\data\\eda.json'}

## 3. Load Qasper

Load `allenai/qasper`, record the split sizes, and save a local dataset copy.

In [3]:
dataset: DatasetDict = load_dataset("allenai/qasper", cache_dir=str(HF_CACHE_DIR))

split_sizes = {split: ds.num_rows for split, ds in dataset.items()}
split_features = {split: list(ds.features.keys()) for split, ds in dataset.items()}

if HF_LOCAL_DATASET_DIR.exists():
    shutil.rmtree(HF_LOCAL_DATASET_DIR)
dataset.save_to_disk(str(HF_LOCAL_DATASET_DIR))

download_metadata = {
    "source": "allenai/qasper",
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    "split_sizes": split_sizes,
    "split_features": split_features,
    "local_dataset_dir": str(HF_LOCAL_DATASET_DIR),
}

print(json.dumps(download_metadata, indent=2))

README.md: 0.00B [00:00, ?B/s]

c:\Users\yassi\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yassi\.cache\huggingface\hub\datasets--allenai--qasper. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


qasper.py: 0.00B [00:00, ?B/s]

qasper/train/0000.parquet:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

qasper/validation/0000.parquet:   0%|          | 0.00/4.75M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/416 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/888 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/281 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/416 [00:00<?, ? examples/s]

{
  "source": "allenai/qasper",
  "downloaded_at_utc": "2026-04-15T15:47:26.424051+00:00",
  "split_sizes": {
    "train": 888,
    "validation": 281,
    "test": 416
  },
  "split_features": {
    "train": [
      "id",
      "title",
      "abstract",
      "full_text",
      "qas",
      "figures_and_tables"
    ],
    "validation": [
      "id",
      "title",
      "abstract",
      "full_text",
      "qas",
      "figures_and_tables"
    ],
    "test": [
      "id",
      "title",
      "abstract",
      "full_text",
      "qas",
      "figures_and_tables"
    ]
  },
  "local_dataset_dir": "C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\data\\raw\\qasper_hf_dataset"
}


## 4. Save the raw splits

Write each untouched split to Parquet and JSONL before applying transformations.

In [4]:
raw_artifacts = []
for split, ds in dataset.items():
    parquet_path = RAW_DIR / f"qasper_{split}.parquet"
    jsonl_path = RAW_DIR / f"qasper_{split}.jsonl"
    ds.to_parquet(str(parquet_path))
    ds.to_json(str(jsonl_path))
    raw_artifacts.append(
        {
            "split": split,
            "rows": int(ds.num_rows),
            "parquet": str(parquet_path),
            "jsonl": str(jsonl_path),
        }
    )

raw_artifacts_df = pd.DataFrame(raw_artifacts)
raw_artifacts_df

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

,split,rows,parquet,jsonl
0,train,888,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...
1,validation,281,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...
2,test,416,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...


## 5. Flatten the nested records

Convert papers, questions, answers, and evidence into tabular records.

In [5]:
WORD_RE = re.compile(r"\b\w+\b", re.UNICODE)
BOOLEAN_VALUES = {"yes", "no", "true", "false"}


def safe_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    return str(value).strip()


def normalize_whitespace(text: str) -> str:
    text = unicodedata.normalize("NFKC", safe_text(text))
    return re.sub(r"\s+", " ", text).strip()


def flatten_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return normalize_whitespace(value)
    if isinstance(value, dict):
        parts = []
        for v in value.values():
            flattened = flatten_text(v)
            if flattened:
                parts.append(flattened)
        return normalize_whitespace(" ".join(parts))
    if isinstance(value, (list, tuple)):
        parts = []
        for item in value:
            flattened = flatten_text(item)
            if flattened:
                parts.append(flattened)
        return normalize_whitespace(" ".join(parts))
    return normalize_whitespace(str(value))


def dict_of_lists_to_records(obj: Dict[str, Any]) -> List[Dict[str, Any]]:
    list_keys = [k for k, v in obj.items() if isinstance(v, list)]
    if not list_keys:
        return [obj]
    max_len = max(len(obj[k]) for k in list_keys) if list_keys else 0
    records: List[Dict[str, Any]] = []
    for i in range(max_len):
        record: Dict[str, Any] = {}
        for k, v in obj.items():
            if isinstance(v, list):
                record[k] = v[i] if i < len(v) else None
            else:
                record[k] = v
        records.append(record)
    return records


def normalize_records(value: Any) -> List[Dict[str, Any]]:
    if value is None:
        return []
    if isinstance(value, dict):
        return dict_of_lists_to_records(value)
    if isinstance(value, list):
        normalized = []
        for item in value:
            if isinstance(item, dict):
                normalized.append(item)
            else:
                normalized.append({"value": item})
        return normalized
    return [{"value": value}]


def build_context_text(example: Dict[str, Any]) -> str:
    parts = [
        safe_text(example.get("title")),
        flatten_text(example.get("abstract")),
        flatten_text(example.get("full_text")),
    ]
    return "\n\n".join([p for p in parts if p]).strip()


def extract_questions(example: Dict[str, Any]) -> List[Dict[str, Any]]:
    qas_raw = example.get("qas")
    qas_records = normalize_records(qas_raw)
    questions = []
    for i, qa in enumerate(qas_records):
        question_text = safe_text(qa.get("question"))
        question_id = safe_text(qa.get("question_id")) or f"q_{i}"
        answers_raw = qa.get("answers") if "answers" in qa else qa.get("answer")
        answers_records = normalize_records(answers_raw)
        if not answers_records:
            answers_records = [{}]
        questions.append(
            {
                "question_id": question_id,
                "question_text": question_text,
                "answers": answers_records,
            }
        )
    return questions


def parse_answer_annotation(annotation: Dict[str, Any], context_text: str) -> Dict[str, Any]:
    ann = annotation if isinstance(annotation, dict) else {"answer": annotation}
    answer_payload = ann.get("answer", ann)
    if not isinstance(answer_payload, dict):
        answer_payload = {"free_form_answer": safe_text(answer_payload)}

    extractive_spans = answer_payload.get("extractive_spans", []) or []
    if isinstance(extractive_spans, str):
        extractive_spans = [extractive_spans]
    extractive_spans = [normalize_whitespace(x) for x in extractive_spans if safe_text(x)]

    free_form_answer = normalize_whitespace(answer_payload.get("free_form_answer", ""))
    yes_no = normalize_whitespace(answer_payload.get("yes_no", "")).lower()
    unanswerable = bool(answer_payload.get("unanswerable", False))

    evidence = ann.get("evidence", [])
    if isinstance(evidence, str):
        evidence = [evidence]
    evidence = [normalize_whitespace(x) for x in evidence if safe_text(x)]
    evidence_text = normalize_whitespace(" ".join(evidence))

    if unanswerable:
        answer_type = "unanswerable"
        answer_text = ""
    elif yes_no in {"yes", "no"}:
        answer_type = "boolean"
        answer_text = yes_no
    elif extractive_spans and free_form_answer:
        answer_type = "hybrid"
        answer_text = free_form_answer
    elif extractive_spans:
        answer_type = "extractive"
        answer_text = normalize_whitespace(" ".join(extractive_spans))
    elif free_form_answer:
        answer_type = "boolean" if free_form_answer.lower() in BOOLEAN_VALUES else "abstractive"
        answer_text = free_form_answer
    else:
        answer_type = "other"
        answer_text = ""

    answer_l = answer_text.lower()
    context_l = safe_text(context_text).lower()
    evidence_l = evidence_text.lower()

    return {
        "annotation_id": safe_text(ann.get("annotation_id")),
        "worker_id": safe_text(ann.get("worker_id")),
        "answer_text": answer_text,
        "answer_type_initial": answer_type,
        "yes_no": yes_no,
        "is_unanswerable": unanswerable,
        "extractive_spans": extractive_spans,
        "evidence_text": evidence_text,
        "has_evidence": bool(evidence_text),
        "answer_in_context": bool(answer_l and (answer_l in context_l)),
        "answer_in_evidence": bool(answer_l and evidence_l and (answer_l in evidence_l)),
    }


def word_count(text: str) -> int:
    return len(WORD_RE.findall(safe_text(text).lower()))


def sentence_count(text: str) -> int:
    txt = safe_text(text).strip()
    if not txt:
        return 0
    chunks = re.split(r"(?<=[.!?])\s+", txt)
    return len([c for c in chunks if c.strip()])


def unique_token_ratio(text: str) -> float:
    tokens = WORD_RE.findall(safe_text(text).lower())
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)


def punctuation_density(text: str) -> float:
    txt = safe_text(text)
    if not txt:
        return 0.0
    punct_count = len(re.findall(r"[\.,;:!?\-\(\)\[\]{}]", txt))
    return punct_count / max(len(txt), 1)


def printable_ratio(text: str) -> float:
    txt = safe_text(text)
    if not txt:
        return 1.0
    printable = sum(ch.isprintable() for ch in txt)
    return printable / len(txt)


def summarize_numeric(series: pd.Series) -> Dict[str, float]:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return {
            "count": 0,
            "mean": 0.0,
            "median": 0.0,
            "std": 0.0,
            "min": 0.0,
            "q1": 0.0,
            "q3": 0.0,
            "p90": 0.0,
            "p95": 0.0,
            "p99": 0.0,
            "max": 0.0,
            "iqr_outlier_threshold": 0.0,
        }
    q1 = float(s.quantile(0.25))
    q3 = float(s.quantile(0.75))
    iqr = q3 - q1
    return {
        "count": int(s.shape[0]),
        "mean": float(s.mean()),
        "median": float(s.median()),
        "std": float(s.std(ddof=0)),
        "min": float(s.min()),
        "q1": q1,
        "q3": q3,
        "p90": float(s.quantile(0.90)),
        "p95": float(s.quantile(0.95)),
        "p99": float(s.quantile(0.99)),
        "max": float(s.max()),
        "iqr_outlier_threshold": float(q3 + 1.5 * iqr),
    }


print("Flattening helpers initialized.")

Flattening helpers initialized.


## 6. Export CSV tables

Save the train, validation, and test tables along with the paper, question, and answer tables.

In [6]:
papers_all = []
qa_all = []
answers_all = []
export_manifest = []

for split, ds in dataset.items():
    split_papers = []
    split_qa = []
    split_answers = []

    for row_idx, example in enumerate(ds):
        paper_id = safe_text(example.get("id")) or f"{split}_paper_{row_idx}"
        title = normalize_whitespace(example.get("title", ""))
        abstract_text = flatten_text(example.get("abstract"))
        full_text = flatten_text(example.get("full_text"))
        context_text = build_context_text(example)

        split_papers.append(
            {
                "split": split,
                "paper_id": paper_id,
                "title": title,
                "abstract_text": abstract_text,
                "full_text": full_text,
                "context_text": context_text,
            }
        )

        questions = extract_questions(example)
        for q_idx, q in enumerate(questions):
            question_id = safe_text(q.get("question_id")) or f"q_{q_idx}"
            question_text = normalize_whitespace(q.get("question_text", ""))
            question_uid = f"{paper_id}::{question_id}"
            answers = q.get("answers", []) or [{}]

            split_qa.append(
                {
                    "split": split,
                    "paper_id": paper_id,
                    "question_id": question_id,
                    "question_uid": question_uid,
                    "question_text": question_text,
                    "context_text": context_text,
                    "answer_count_expected": len(answers),
                }
            )

            for a_idx, ann in enumerate(answers):
                parsed = parse_answer_annotation(ann, context_text)
                answer_id = f"{question_uid}::a_{a_idx}"
                split_answers.append(
                    {
                        "split": split,
                        "paper_id": paper_id,
                        "question_id": question_id,
                        "question_uid": question_uid,
                        "answer_id": answer_id,
                        "question_text": question_text,
                        "context_text": context_text,
                        **parsed,
                    }
                )

    papers_df_split = pd.DataFrame(split_papers).drop_duplicates(subset=["paper_id"])
    qa_df_split = pd.DataFrame(split_qa).drop_duplicates(subset=["question_uid"])
    answers_df_split = pd.DataFrame(split_answers)

    split_main_csv = CSV_DIR / f"{split}.csv"
    split_papers_csv = CSV_DIR / f"{split}_papers.csv"
    split_qa_csv = CSV_DIR / f"{split}_qa.csv"
    split_answers_csv = CSV_DIR / f"{split}_answers.csv"

    answers_df_split.to_csv(split_main_csv, index=False)
    papers_df_split.to_csv(split_papers_csv, index=False)
    qa_df_split.to_csv(split_qa_csv, index=False)
    answers_df_split.to_csv(split_answers_csv, index=False)

    papers_df_split.to_parquet(PROCESSED_DIR / f"{split}_papers.parquet", index=False)
    qa_df_split.to_parquet(PROCESSED_DIR / f"{split}_qa.parquet", index=False)
    answers_df_split.to_parquet(PROCESSED_DIR / f"{split}_answers.parquet", index=False)

    export_manifest.extend(
        [
            {"split": split, "artifact": "main_csv", "path": str(split_main_csv), "rows": int(answers_df_split.shape[0])},
            {"split": split, "artifact": "papers_csv", "path": str(split_papers_csv), "rows": int(papers_df_split.shape[0])},
            {"split": split, "artifact": "qa_csv", "path": str(split_qa_csv), "rows": int(qa_df_split.shape[0])},
            {"split": split, "artifact": "answers_csv", "path": str(split_answers_csv), "rows": int(answers_df_split.shape[0])},
        ]
    )

    papers_all.append(papers_df_split)
    qa_all.append(qa_df_split)
    answers_all.append(answers_df_split)

papers_df = pd.concat(papers_all, ignore_index=True)
qa_df = pd.concat(qa_all, ignore_index=True)
answers_df = pd.concat(answers_all, ignore_index=True)

papers_df.to_csv(CSV_DIR / "all_papers.csv", index=False)
qa_df.to_csv(CSV_DIR / "all_qa.csv", index=False)
answers_df.to_csv(CSV_DIR / "all_answers.csv", index=False)

export_manifest_df = pd.DataFrame(export_manifest)
print(f"papers_df shape: {papers_df.shape}")
print(f"qa_df shape: {qa_df.shape}")
print(f"answers_df shape: {answers_df.shape}")
export_manifest_df.head(12)

papers_df shape: (1585, 6)
qa_df shape: (5049, 7)
answers_df shape: (7993, 18)


,split,artifact,path,rows
0,train,main_csv,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,2675
1,train,papers_csv,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,888
2,train,qa_csv,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,2593
3,train,answers_csv,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,2675
4,validation,main_csv,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,1764
5,validation,papers_csv,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,281
6,validation,qa_csv,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,1005
7,validation,answers_csv,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,1764
8,test,main_csv,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,3554
9,test,papers_csv,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,416


## 7. Check the schema and missing values

Inspect column types, missing values, unique values, duplicate rows, and malformed records.

In [9]:
def make_hashable(value: Any) -> Any:
    if isinstance(value, (list, dict, tuple, set)):
        try:
            return json.dumps(value, sort_keys=True, ensure_ascii=False)
        except TypeError:
            return str(value)
    return value


def safe_nunique(series: pd.Series) -> int:
    return int(series.map(make_hashable).nunique(dropna=True))


def schema_audit(df: pd.DataFrame, table_name: str) -> Dict[str, Any]:
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    cardinality = df.apply(safe_nunique, axis=0).sort_values(ascending=False)

    hashable_df = df.apply(lambda col: col.map(make_hashable))
    duplicate_rows = int(hashable_df.duplicated().sum())

    empty_row_mask = df.fillna("").astype(str).apply(lambda c: c.str.strip()).eq("").all(axis=1)
    malformed_empty_rows = int(empty_row_mask.sum())

    high_null = missing_pct[missing_pct >= 30.0]
    high_null_dict = {k: float(v) for k, v in high_null.items()}

    return {
        "table": table_name,
        "shape": {"rows": int(df.shape[0]), "cols": int(df.shape[1])},
        "dtypes": {k: str(v) for k, v in df.dtypes.items()},
        "missing_pct": {k: float(v) for k, v in missing_pct.items()},
        "cardinality": {k: int(v) for k, v in cardinality.items()},
        "duplicate_rows": duplicate_rows,
        "malformed_empty_rows": malformed_empty_rows,
        "high_null_fields_ge_30pct": high_null_dict,
    }


schema_summary = {
    "papers": schema_audit(papers_df, "papers"),
    "qa": schema_audit(qa_df, "qa"),
    "answers": schema_audit(answers_df, "answers"),
}

missingness_summary_df = pd.DataFrame(
    {
        "papers_missing_pct": pd.Series(schema_summary["papers"]["missing_pct"]),
        "qa_missing_pct": pd.Series(schema_summary["qa"]["missing_pct"]),
        "answers_missing_pct": pd.Series(schema_summary["answers"]["missing_pct"]),
    }
).fillna(0.0)

missingness_summary_df.sort_values(by="answers_missing_pct", ascending=False).head(20)

,papers_missing_pct,qa_missing_pct,answers_missing_pct
abstract_text,0.0,0.0,0.0
annotation_id,0.0,0.0,0.0
answer_count_expected,0.0,0.0,0.0
answer_id,0.0,0.0,0.0
answer_in_context,0.0,0.0,0.0
answer_in_evidence,0.0,0.0,0.0
answer_text,0.0,0.0,0.0
answer_type_initial,0.0,0.0,0.0
context_text,0.0,0.0,0.0
evidence_text,0.0,0.0,0.0


## 8. Record counts by split

Count papers, questions, answers, annotators, and answers per question for each split.

In [10]:
volume_metrics: Dict[str, Dict[str, Any]] = {}

for split in sorted(answers_df["split"].dropna().unique().tolist()):
    split_papers = papers_df[papers_df["split"] == split]
    split_qa = qa_df[qa_df["split"] == split]
    split_answers = answers_df[answers_df["split"] == split]

    qa_per_paper = split_qa.groupby("paper_id")["question_uid"].nunique()
    answers_per_question_split = split_answers.groupby("question_uid")["answer_id"].nunique()

    annotator_count = int(
        split_answers["worker_id"].replace("", np.nan).dropna().nunique()
    ) if "worker_id" in split_answers.columns else 0

    volume_metrics[split] = {
        "papers": int(split_papers["paper_id"].nunique()),
        "questions": int(split_qa["question_uid"].nunique()),
        "answers": int(split_answers["answer_id"].nunique()),
        "unique_annotators": annotator_count,
        "qa_per_paper": summarize_numeric(qa_per_paper),
        "answers_per_question": summarize_numeric(answers_per_question_split),
    }

volume_metrics_df = pd.DataFrame(
    {
        split: {
            "papers": m["papers"],
            "questions": m["questions"],
            "answers": m["answers"],
            "unique_annotators": m["unique_annotators"],
            "qa_per_paper_median": m["qa_per_paper"]["median"],
            "answers_per_question_median": m["answers_per_question"]["median"],
        }
        for split, m in volume_metrics.items()
    }
).T

volume_metrics_df

,papers,questions,answers,unique_annotators,qa_per_paper_median,answers_per_question_median
test,416.0,1451.0,3554.0,66.0,3.0,2.0
train,888.0,2593.0,2675.0,57.0,3.0,1.0
validation,281.0,1005.0,1764.0,41.0,3.0,2.0


## 9. Text length statistics

Measure question, answer, context, and evidence lengths and calculate IQR outlier limits.

In [11]:
for col in ["question_text", "context_text"]:
    qa_df[f"{col}_char_len"] = qa_df[col].fillna("").astype(str).map(len)
    qa_df[f"{col}_word_len"] = qa_df[col].fillna("").astype(str).map(word_count)

for col in ["answer_text", "evidence_text", "context_text", "question_text"]:
    answers_df[f"{col}_char_len"] = answers_df[col].fillna("").astype(str).map(len)
    answers_df[f"{col}_word_len"] = answers_df[col].fillna("").astype(str).map(word_count)

text_length_stats = {
    "question_char_len": summarize_numeric(qa_df["question_text_char_len"]),
    "question_word_len": summarize_numeric(qa_df["question_text_word_len"]),
    "answer_char_len": summarize_numeric(answers_df["answer_text_char_len"]),
    "answer_word_len": summarize_numeric(answers_df["answer_text_word_len"]),
    "context_char_len": summarize_numeric(qa_df["context_text_char_len"]),
    "context_word_len": summarize_numeric(qa_df["context_text_word_len"]),
    "evidence_char_len": summarize_numeric(answers_df["evidence_text_char_len"]),
    "evidence_word_len": summarize_numeric(answers_df["evidence_text_word_len"]),
}

text_length_summary_df = pd.DataFrame(text_length_stats).T
text_length_summary_df

,count,mean,median,std,min,q1,q3,p90,p95,p99,max,iqr_outlier_threshold
question_char_len,5049.0,50.482076,45.0,21.289973,4.0,36.0,60.0,79.0,92.0,120.0,176.0,96.0
question_word_len,5049.0,8.328580,8.0,3.305346,1.0,6.0,10.0,13.0,15.0,20.0,26.0,16.0
answer_char_len,7993.0,74.696860,40.0,114.809479,0.0,3.0,102.0,185.8,261.0,492.0,2223.0,250.5
answer_word_len,7993.0,11.848868,6.0,18.459410,0.0,1.0,16.0,29.0,42.0,80.0,363.0,38.5
context_char_len,5049.0,24489.036839,23782.0,12383.166527,1069.0,16188.0,29180.0,35981.4,43808.0,70808.0,170230.0,48668.0
context_word_len,5049.0,3848.211923,3711.0,1945.184427,159.0,2560.0,4584.0,5655.4,6910.0,10869.0,27248.0,7620.0
evidence_char_len,7993.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
evidence_word_len,7993.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 10. Token lengths and context budgets

Tokenize the text with `bert-base-uncased` and estimate how much context is retained at several token limits.

In [12]:
TOKENIZER_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)


@lru_cache(maxsize=400_000)
def token_length(text: str) -> int:
    txt = safe_text(text)
    if not txt:
        return 0
    return len(tokenizer.encode(txt, add_special_tokens=False, truncation=False))


qa_df["question_text_tokens"] = qa_df["question_text"].fillna("").astype(str).map(token_length)
qa_df["context_text_tokens"] = qa_df["context_text"].fillna("").astype(str).map(token_length)
answers_df["answer_text_tokens"] = answers_df["answer_text"].fillna("").astype(str).map(token_length)
answers_df["evidence_text_tokens"] = answers_df["evidence_text"].fillna("").astype(str).map(token_length)
answers_df["question_text_tokens"] = answers_df["question_text"].fillna("").astype(str).map(token_length)
answers_df["context_text_tokens"] = answers_df["context_text"].fillna("").astype(str).map(token_length)

budget_limits = [256, 512, 1024]

tokenizer_budget_analysis = {
    "question_text_tokens": {
        "summary": summarize_numeric(qa_df["question_text_tokens"]),
        "exceed_ratio": {str(b): float((qa_df["question_text_tokens"] > b).mean()) for b in budget_limits},
    },
    "answer_text_tokens": {
        "summary": summarize_numeric(answers_df["answer_text_tokens"]),
        "exceed_ratio": {str(b): float((answers_df["answer_text_tokens"] > b).mean()) for b in budget_limits},
    },
    "context_text_tokens": {
        "summary": summarize_numeric(qa_df["context_text_tokens"]),
        "exceed_ratio": {str(b): float((qa_df["context_text_tokens"] > b).mean()) for b in budget_limits},
    },
    "evidence_text_tokens": {
        "summary": summarize_numeric(answers_df["evidence_text_tokens"]),
        "exceed_ratio": {str(b): float((answers_df["evidence_text_tokens"] > b).mean()) for b in budget_limits},
    },
}

pd.DataFrame({k: v["exceed_ratio"] for k, v in tokenizer_budget_analysis.items()}).T

Token indices sequence length is longer than the specified maximum sequence length for this model (4348 > 512). Running this sequence through the model will result in indexing errors


,256,512,1024
question_text_tokens,0.000000,0.000000,0.000000
answer_text_tokens,0.001501,0.000125,0.000000
context_text_tokens,0.999406,0.999010,0.997821
evidence_text_tokens,0.000000,0.000000,0.000000


## 11. Answers per question

Measure the number of answers per question and compare answer length, type, and lexical overlap.

In [13]:
answers_per_question = answers_df.groupby("question_uid")["answer_id"].nunique()


def token_set(text: str) -> set:
    return set(WORD_RE.findall(safe_text(text).lower()))


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a.intersection(b)) / len(a.union(b))


variability_records = []
for question_uid, group in answers_df.groupby("question_uid"):
    if group.shape[0] < 2:
        continue

    answer_texts = group["answer_text"].fillna("").astype(str).tolist()
    answer_types = group["answer_type_initial"].fillna("other").astype(str).tolist()
    lengths = group["answer_text_word_len"].fillna(0).astype(float).tolist()

    sets = [token_set(t) for t in answer_texts]
    overlaps = []
    for i in range(len(sets)):
        for j in range(i + 1, len(sets)):
            overlaps.append(jaccard(sets[i], sets[j]))

    variability_records.append(
        {
            "question_uid": question_uid,
            "n_answers": int(group.shape[0]),
            "answer_length_std": float(np.std(lengths, ddof=0)),
            "answer_type_unique": int(len(set(answer_types))),
            "mean_pairwise_jaccard": float(np.mean(overlaps) if overlaps else 1.0),
        }
    )

variability_df = pd.DataFrame(variability_records)
if variability_df.empty:
    variability_summary = {
        "pairwise_variability_questions": 0,
        "mean_answer_length_std": 0.0,
        "mean_pairwise_jaccard": 0.0,
        "ambiguous_questions_estimate": 0,
    }
else:
    ambiguous_mask = (
        (variability_df["mean_pairwise_jaccard"] < 0.25)
        | (variability_df["answer_type_unique"] > 1)
        | (variability_df["answer_length_std"] > variability_df["answer_length_std"].quantile(0.90))
    )
    variability_summary = {
        "pairwise_variability_questions": int(variability_df.shape[0]),
        "mean_answer_length_std": float(variability_df["answer_length_std"].mean()),
        "mean_pairwise_jaccard": float(variability_df["mean_pairwise_jaccard"].mean()),
        "ambiguous_questions_estimate": int(ambiguous_mask.sum()),
    }

variability_summary

{'pairwise_variability_questions': 2253,
 'mean_answer_length_std': 4.932124879836659,
 'mean_pairwise_jaccard': 0.47458561918496023,
 'ambiguous_questions_estimate': 1233}

## 12. Approximate answer types

Assign heuristic answer types using boolean values and overlap with the paper context or evidence.

In [14]:
def classify_answer_type(row: pd.Series) -> str:
    if bool(row.get("is_unanswerable", False)):
        return "unanswerable"

    answer_text = normalize_whitespace(row.get("answer_text", "")).lower()
    if answer_text in {"yes", "no", "true", "false"}:
        return "boolean"

    in_evidence = bool(row.get("answer_in_evidence", False))
    in_context = bool(row.get("answer_in_context", False))
    answer_tokens = word_count(answer_text)

    if answer_text and (in_evidence or in_context):
        if answer_tokens <= 40:
            return "extractive"
        return "hybrid"

    if answer_tokens == 0:
        return "other"

    return "abstractive"


answers_df["answer_type"] = answers_df.apply(classify_answer_type, axis=1)

answer_type_distribution = (
    answers_df.groupby(["split", "answer_type"]).size().rename("count").reset_index()
)

answer_type_balance = (
    answer_type_distribution.pivot(index="split", columns="answer_type", values="count")
    .fillna(0)
    .astype(int)
)

answer_type_balance

answer_type,abstractive,extractive,hybrid,other,unanswerable
split,,,,,
test,1607,1045,43,493,366
train,1119,810,56,409,281
validation,813,566,14,208,163


## 13. Context statistics

Measure sentence length, token diversity, punctuation density, and printable-character ratio.

In [15]:
context_profile_df = qa_df[["split", "question_uid", "context_text"]].drop_duplicates().copy()
context_profile_df["context_sentence_count"] = context_profile_df["context_text"].fillna("").astype(str).map(sentence_count)
context_profile_df["context_word_len"] = context_profile_df["context_text"].fillna("").astype(str).map(word_count)
context_profile_df["context_avg_sentence_len"] = context_profile_df["context_word_len"] / context_profile_df["context_sentence_count"].replace(0, np.nan)
context_profile_df["context_avg_sentence_len"] = context_profile_df["context_avg_sentence_len"].fillna(0.0)
context_profile_df["context_unique_token_ratio"] = context_profile_df["context_text"].fillna("").astype(str).map(unique_token_ratio)
context_profile_df["context_punctuation_density"] = context_profile_df["context_text"].fillna("").astype(str).map(punctuation_density)
context_profile_df["context_printable_ratio"] = context_profile_df["context_text"].fillna("").astype(str).map(printable_ratio)

context_complexity_stats = {
    "context_sentence_count": summarize_numeric(context_profile_df["context_sentence_count"]),
    "context_avg_sentence_len": summarize_numeric(context_profile_df["context_avg_sentence_len"]),
    "context_unique_token_ratio": summarize_numeric(context_profile_df["context_unique_token_ratio"]),
    "context_punctuation_density": summarize_numeric(context_profile_df["context_punctuation_density"]),
    "context_printable_ratio": summarize_numeric(context_profile_df["context_printable_ratio"]),
}

context_profile_df[[
    "context_sentence_count",
    "context_avg_sentence_len",
    "context_unique_token_ratio",
    "context_punctuation_density",
    "context_printable_ratio",
]].describe(percentiles=[0.5, 0.9, 0.95]).T

,count,mean,std,min,50%,90%,95%,max
context_sentence_count,5049.0,173.410378,89.350196,7.000000,163.000000,255.000000,311.000000,1381.000000
context_avg_sentence_len,5049.0,22.437149,3.105904,10.035944,22.177215,26.269231,27.783356,39.305556
context_unique_token_ratio,5049.0,0.255327,0.051710,0.077767,0.249693,0.323189,0.337854,0.637555
context_punctuation_density,5049.0,0.027674,0.007231,0.012656,0.026268,0.036690,0.041028,0.070444
context_printable_ratio,5049.0,0.999787,0.000266,0.993340,0.999832,0.999889,0.999909,0.999977


## 14. Flag noisy records

Mark empty fields, unusual lengths, duplicate question-answer pairs, low printable ratios, and suspicious Unicode characters.

In [16]:
def norm_key(text: str) -> str:
    return re.sub(r"\s+", " ", safe_text(text).lower()).strip()


answers_df["norm_question"] = answers_df["question_text"].fillna("").astype(str).map(norm_key)
answers_df["norm_answer"] = answers_df["answer_text"].fillna("").astype(str).map(norm_key)
answers_df["context_printable_ratio"] = answers_df["context_text"].fillna("").astype(str).map(printable_ratio)

question_word_p99 = float(answers_df["question_text_word_len"].quantile(0.99)) if not answers_df.empty else 0.0
answer_word_p99 = float(answers_df["answer_text_word_len"].quantile(0.99)) if not answers_df.empty else 0.0
context_token_p99 = float(answers_df["context_text_tokens"].quantile(0.99)) if not answers_df.empty else 0.0

answers_df["flag_empty_question"] = answers_df["question_text_word_len"] == 0
answers_df["flag_empty_answer_non_unanswerable"] = (answers_df["answer_text_word_len"] == 0) & (~answers_df["is_unanswerable"].astype(bool))
answers_df["flag_too_short_question"] = answers_df["question_text_word_len"] < 3
answers_df["flag_too_long_question"] = answers_df["question_text_word_len"] > question_word_p99
answers_df["flag_too_long_answer"] = answers_df["answer_text_word_len"] > answer_word_p99
answers_df["flag_too_long_context"] = answers_df["context_text_tokens"] > context_token_p99
answers_df["flag_low_printable_context"] = answers_df["context_printable_ratio"] < 0.95
answers_df["flag_suspicious_unicode"] = answers_df["context_text"].fillna("").astype(str).str.contains(r"[\uFFFD\u200B\u2060]", regex=True)
answers_df["flag_duplicate_qa_pair"] = answers_df.duplicated(subset=["split", "paper_id", "norm_question", "norm_answer"], keep=False)
answers_df["flag_extractive_not_in_context"] = (answers_df["answer_type"] == "extractive") & (~answers_df["answer_in_context"].astype(bool))

anomaly_columns = [
    "flag_empty_question",
    "flag_empty_answer_non_unanswerable",
    "flag_too_short_question",
    "flag_too_long_question",
    "flag_too_long_answer",
    "flag_too_long_context",
    "flag_low_printable_context",
    "flag_suspicious_unicode",
    "flag_duplicate_qa_pair",
    "flag_extractive_not_in_context",
]

answers_df["anomaly_score"] = answers_df[anomaly_columns].sum(axis=1)

anomaly_counts = {col: int(answers_df[col].sum()) for col in anomaly_columns}
anomaly_counts["rows_with_any_flag"] = int((answers_df["anomaly_score"] > 0).sum())

anomaly_samples_df = answers_df[answers_df["anomaly_score"] > 0][
    ["split", "paper_id", "question_uid", "answer_id", "question_text", "answer_text", "answer_type", "anomaly_score"] + anomaly_columns
].head(25)

anomaly_samples_df

,split,paper_id,question_uid,answer_id,question_text,answer_text,answer_type,anomaly_score,flag_empty_question,flag_empty_answer_non_unanswerable,flag_too_short_question,flag_too_long_question,flag_too_long_answer,flag_too_long_context,flag_low_printable_context,flag_suspicious_unicode,flag_duplicate_qa_pair,flag_extractive_not_in_context
2,train,1909.00694,1909.00694::9d578ddccc27dd849244d632dd0f6bf273...,1909.00694::9d578ddccc27dd849244d632dd0f6bf273...,What are the results?,Using all data to train: AL -- BiGRU achieved ...,abstractive,1,False,False,False,False,True,False,False,False,False,False
8,train,1909.00694,1909.00694::c029deb7f99756d2669abad0a349d91742...,1909.00694::c029deb7f99756d2669abad0a349d91742...,How big are improvements of supervszed learnin...,3%,abstractive,1,False,False,False,True,False,False,False,False,False,False
12,train,2003.07723,2003.07723::3a9d391d25cde8af3334ac62d478b36b30...,2003.07723::3a9d391d25cde8af3334ac62d478b36b30...,Does the paper report macro F1?,,other,2,False,True,False,False,False,False,False,False,True,False
13,train,2003.07723,2003.07723::3a9d391d25cde8af3334ac62d478b36b30...,2003.07723::3a9d391d25cde8af3334ac62d478b36b30...,Does the paper report macro F1?,,other,2,False,True,False,False,False,False,False,False,True,False
16,train,1705.09665,1705.09665::003f884d3893532f8c302431c9f70be6f6...,1705.09665::003f884d3893532f8c302431c9f70be6f6...,Do they report results only on English data?,,other,2,False,True,False,False,False,False,False,False,True,False
17,train,1705.09665,1705.09665::003f884d3893532f8c302431c9f70be6f6...,1705.09665::003f884d3893532f8c302431c9f70be6f6...,Do they report results only on English data?,,unanswerable,1,False,False,False,False,False,False,False,False,True,False
18,train,1705.09665,1705.09665::bb97537a0a7c8f12a3f65eba73cefa6abc...,1705.09665::bb97537a0a7c8f12a3f65eba73cefa6abc...,How do the various social phenomena examined m...,Dynamic communities have substantially higher ...,abstractive,1,False,False,False,False,True,False,False,False,False,False
42,train,1811.00942,1811.00942::e07df8f613dbd567a35318cd6f6f4cb959...,1811.00942::e07df8f613dbd567a35318cd6f6f4cb959...,What is a commonly used evaluation metric for ...,perplexity,extractive,1,False,False,False,False,False,False,False,False,True,False
43,train,1811.00942,1811.00942::e07df8f613dbd567a35318cd6f6f4cb959...,1811.00942::e07df8f613dbd567a35318cd6f6f4cb959...,What is a commonly used evaluation metric for ...,perplexity,extractive,1,False,False,False,False,False,False,False,False,True,False
46,train,1805.02400,1805.02400::98b11f70239ef0e22511a3ecf6e413ecb7...,1805.02400::98b11f70239ef0e22511a3ecf6e413ecb7...,Do they use a pretrained NMT model to help gen...,,other,2,False,True,False,False,False,False,False,False,True,False


## 15. Plots

Save the main distributions, split comparisons, correlations, and anomaly counts under `reports/figures/`.

In [17]:
figure_paths: Dict[str, str] = {}


def save_current_figure(file_name: str) -> str:
    out_path = FIGURES_DIR / file_name
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    return str(out_path)


# 1) Question token length histogram
plt.figure()
sns.histplot(qa_df["question_text_tokens"], bins=50, kde=True)
plt.title("Question Token Length Distribution")
plt.xlabel("Tokens")
plt.ylabel("Count")
figure_paths["question_token_histogram"] = save_current_figure("question_token_histogram.png")

# 2) Context token boxplot by split
plt.figure()
sns.boxplot(data=qa_df, x="split", y="context_text_tokens")
plt.title("Context Token Length by Split")
plt.xlabel("Split")
plt.ylabel("Context Tokens")
figure_paths["context_tokens_boxplot"] = save_current_figure("context_tokens_boxplot.png")

# 3) Answer type balance
answer_type_plot_df = answer_type_distribution.copy()
plt.figure()
sns.barplot(data=answer_type_plot_df, x="answer_type", y="count", hue="split")
plt.title("Answer Type Distribution by Split")
plt.xlabel("Answer Type")
plt.ylabel("Count")
plt.xticks(rotation=25)
figure_paths["answer_type_balance"] = save_current_figure("answer_type_balance.png")

# 4) Answers per question distribution
plt.figure()
sns.histplot(answers_per_question, bins=30, kde=False)
plt.title("Answers per Question Distribution")
plt.xlabel("Answers per Question")
plt.ylabel("Questions")
figure_paths["answers_per_question_histogram"] = save_current_figure("answers_per_question_histogram.png")

# 5) Correlation heatmap
numeric_for_corr = answers_df[
    [
        "question_text_word_len",
        "answer_text_word_len",
        "context_text_word_len",
        "question_text_tokens",
        "answer_text_tokens",
        "context_text_tokens",
        "anomaly_score",
    ]
].copy()

corr = numeric_for_corr.corr(numeric_only=True)
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", square=True)
plt.title("Correlation Heatmap for Numeric Features")
figure_paths["correlation_heatmap"] = save_current_figure("correlation_heatmap.png")

# 6) Anomaly counts bar plot
anomaly_plot_df = pd.DataFrame({"rule": list(anomaly_counts.keys()), "count": list(anomaly_counts.values())})
plt.figure(figsize=(11, 5))
sns.barplot(data=anomaly_plot_df, x="rule", y="count")
plt.title("Anomaly Rule Trigger Counts")
plt.xlabel("Rule")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
figure_paths["anomaly_rule_counts"] = save_current_figure("anomaly_rule_counts.png")

figure_paths

c:\Users\yassi\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:632: FutureWarning: SeriesGroupBy.grouper is deprecated and will be removed in a future version of pandas.
  positions = grouped.grouper.result_index.to_numpy(dtype=float)
c:\Users\yassi\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\_base.py:948: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
c:\Users\yassi\AppData\Local\Programs\Python\Python311\Lib\site-packages\seaborn\categorical.py:1273: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping colu

{'question_token_histogram': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\reports\\figures\\question_token_histogram.png',
 'context_tokens_boxplot': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\reports\\figures\\context_tokens_boxplot.png',
 'answer_type_balance': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\reports\\figures\\answer_type_balance.png',
 'answers_per_question_histogram': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\reports\\figures\\answers_per_question_histogram.png',
 'correlation_heatmap': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\reports\\figures\\correlation_heatmap.png',
 'anomaly_rule_counts': 'C:\\Users\\yassi\\Documents\\PFE\\Sciq Finetunning\\reports\\figures\\anomaly_rule_counts.png'}

## 16. Summarize findings for fine-tuning

Summarize the observed context lengths, answer formats, evidence coverage, and difficult samples.

In [18]:
qa_df["question_starter"] = (
    qa_df["question_text"].fillna("").astype(str).str.extract(r"^\W*([A-Za-z]+)", expand=False).str.lower().fillna("unknown")
)

top_question_starters = qa_df["question_starter"].value_counts().head(15).to_dict()

context_quantiles = qa_df["context_text_tokens"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_dict()
question_quantiles = qa_df["question_text_tokens"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_dict()
answer_quantiles = answers_df["answer_text_tokens"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_dict()

recommended_context_budget = int(max(512, min(4096, np.ceil(context_quantiles[0.95] * 1.10))))
recommended_answer_budget = int(max(16, min(512, np.ceil(answer_quantiles[0.95] * 1.20))))

hard_sample_mask = (
    (answers_df["context_text_tokens"] > context_quantiles[0.95])
    | (answers_df["anomaly_score"] > 0)
    | (answers_df["answer_type"].isin(["hybrid", "abstractive"]))
)

prompt_template_recommendation = """You are a scientific question-answering assistant.
Read the context carefully and answer the question concisely and faithfully.

[CONTEXT]
{context}

[QUESTION]
{question}

[ANSWER]
"""

key_insights = {
    "optimal_input_format": {
        "recommended_prompt_template": prompt_template_recommendation,
        "fields": ["context", "question", "answer"],
        "note": "Use explicit section delimiters and keep context/question ordering stable.",
    },
    "typical_context_size_tokens": {
        "p50": float(context_quantiles[0.5]),
        "p75": float(context_quantiles[0.75]),
        "p90": float(context_quantiles[0.9]),
        "p95": float(context_quantiles[0.95]),
        "recommended_budget": recommended_context_budget,
    },
    "answer_style_variability": {
        "answer_type_balance": answer_type_balance.to_dict(orient="index"),
        "answer_tokens_p50": float(answer_quantiles[0.5]),
        "answer_tokens_p95": float(answer_quantiles[0.95]),
        "recommended_answer_budget": recommended_answer_budget,
    },
    "difficulty_estimation": {
        "heuristic": "High context length + high anomaly score + abstractive/hybrid answer",
        "estimated_hard_sample_ratio": float(hard_sample_mask.mean()),
        "estimated_hard_sample_count": int(hard_sample_mask.sum()),
    },
    "question_patterns": {
        "top_question_starters": top_question_starters,
        "question_tokens_p95": float(question_quantiles[0.95]),
    },
}

key_insights

{'optimal_input_format': {'recommended_prompt_template': 'You are a scientific question-answering assistant.\nRead the context carefully and answer the question concisely and faithfully.\n\n[CONTEXT]\n{context}\n\n[QUESTION]\n{question}\n\n[ANSWER]\n',
  'fields': ['context', 'question', 'answer'],
  'note': 'Use explicit section delimiters and keep context/question ordering stable.'},
 'typical_context_size_tokens': {'p50': 5120.0,
  'p75': 6530.0,
  'p90': 8128.0,
  'p95': 9687.999999999993,
  'recommended_budget': 4096},
 'answer_style_variability': {'answer_type_balance': {'test': {'abstractive': 1607,
    'extractive': 1045,
    'hybrid': 43,
    'other': 493,
    'unanswerable': 366},
   'train': {'abstractive': 1119,
    'extractive': 810,
    'hybrid': 56,
    'other': 409,
    'unanswerable': 281},
   'validation': {'abstractive': 813,
    'extractive': 566,
    'hybrid': 14,
    'other': 208,
    'unanswerable': 163}},
  'answer_tokens_p50': 10.0,
  'answer_tokens_p95': 62.0,

## 17. Draft preprocessing steps

List the preprocessing steps used later to build the fine-tuning inputs.

In [19]:
recommended_preprocessing_steps = [
    {
        "step": "normalize_text",
        "description": "Apply Unicode normalization (NFKC), collapse whitespace, and strip control characters.",
    },
    {
        "step": "filter_broken_rows",
        "description": "Remove rows with empty question text and empty non-unanswerable answers.",
    },
    {
        "step": "context_budgeting",
        "description": f"Tokenize and truncate/chunk context to ~{recommended_context_budget} tokens.",
    },
    {
        "step": "evidence_focus",
        "description": "When evidence spans are available, prioritize evidence-centric context windows.",
    },
    {
        "step": "deduplicate",
        "description": "Deduplicate exact question-answer pairs per paper and optionally globally.",
    },
    {
        "step": "answer_type_balance",
        "description": "Monitor class imbalance across extractive/abstractive/boolean/unanswerable and rebalance with sampling.",
    },
    {
        "step": "hard_sample_curriculum",
        "description": "Use anomaly and difficulty proxies to stage training from easy to hard samples.",
    },
    {
        "step": "split_integrity_check",
        "description": "Re-check split-level leakage by paper/question ID before final fine-tuning export.",
    },
]

preprocessing_strategy = {
    "normalization": {
        "unicode": "NFKC",
        "whitespace": "collapse",
        "lowercase": False,
    },
    "filtering": {
        "drop_empty_question": True,
        "drop_empty_answer_non_unanswerable": True,
        "drop_low_printable_context_below": 0.95,
    },
    "tokenization": {
        "tokenizer": TOKENIZER_NAME,
        "max_context_tokens": recommended_context_budget,
        "max_answer_tokens": recommended_answer_budget,
    },
    "sampling": {
        "stratify_by_answer_type": True,
        "include_hard_sample_tag": True,
    },
}

preprocessing_strategy

{'normalization': {'unicode': 'NFKC',
  'whitespace': 'collapse',
  'lowercase': False},
 'filtering': {'drop_empty_question': True,
  'drop_empty_answer_non_unanswerable': True,
  'drop_low_printable_context_below': 0.95},
 'tokenization': {'tokenizer': 'bert-base-uncased',
  'max_context_tokens': 4096,
  'max_answer_tokens': 75},
 'sampling': {'stratify_by_answer_type': True,
  'include_hard_sample_tag': True}}

## 18. Write `eda.json`

Store the statistics, distributions, anomaly counts, and preprocessing notes in one JSON file.

In [20]:
def to_builtin(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    return obj


distributions_payload = {
    "answers_per_question": summarize_numeric(answers_per_question),
    "answer_type_distribution": answer_type_balance.to_dict(orient="index"),
    "top_question_starters": top_question_starters,
    "token_length_exceed_ratio": {
        metric: payload["exceed_ratio"]
        for metric, payload in tokenizer_budget_analysis.items()
    },
}

statistics_payload = {
    "split_sizes": split_sizes,
    "volume_metrics": volume_metrics,
    "text_length_stats": text_length_stats,
    "context_complexity_stats": context_complexity_stats,
    "variability_summary": variability_summary,
    "schema_summary": schema_summary,
}

eda_payload = {
    "statistics": statistics_payload,
    "distributions": distributions_payload,
    "key_insights": key_insights,
    "recommended_preprocessing_steps": recommended_preprocessing_steps,
    "anomalies": {
        "counts": anomaly_counts,
        "sample_rows": anomaly_samples_df.to_dict(orient="records"),
    },
    "tokenizer_budget_analysis": tokenizer_budget_analysis,
    "reproducibility": {
        "seed": SEED,
        "package_versions": PACKAGE_VERSIONS,
        "tokenizer": TOKENIZER_NAME,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    },
    "artifacts": {
        "csv_dir": str(CSV_DIR),
        "raw_dir": str(RAW_DIR),
        "processed_dir": str(PROCESSED_DIR),
        "figures": figure_paths,
    },
}

with open(EDA_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(to_builtin(eda_payload), f, indent=2, ensure_ascii=False)

print(f"Saved EDA JSON to: {EDA_JSON_PATH}")
list(distributions_payload.keys())

Saved EDA JSON to: C:\Users\yassi\Documents\PFE\Sciq Finetunning\data\eda.json


['answers_per_question',
 'answer_type_distribution',
 'top_question_starters',
 'token_length_exceed_ratio']

## 19. Check the generated files

Confirm that the expected variables and output files exist, then record file sizes and SHA-256 hashes.

In [21]:
def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


required_vars = [
    "dataset",
    "papers_df",
    "qa_df",
    "answers_df",
    "tokenizer_budget_analysis",
    "key_insights",
    "recommended_preprocessing_steps",
    "eda_payload",
]
missing_vars = [v for v in required_vars if v not in globals()]
assert not missing_vars, f"Missing expected variables from prior sections: {missing_vars}"

required_files = [EDA_JSON_PATH]
for split in sorted(split_sizes.keys()):
    required_files.extend(
        [
            CSV_DIR / f"{split}.csv",
            CSV_DIR / f"{split}_papers.csv",
            CSV_DIR / f"{split}_qa.csv",
            CSV_DIR / f"{split}_answers.csv",
            RAW_DIR / f"qasper_{split}.parquet",
            RAW_DIR / f"qasper_{split}.jsonl",
        ]
    )
required_files.extend([Path(p) for p in figure_paths.values()])

missing_files = [str(p) for p in required_files if not p.exists()]
assert not missing_files, f"Missing expected output files: {missing_files}"

validation_records = []
for p in required_files:
    validation_records.append(
        {
            "path": str(p),
            "size_bytes": int(p.stat().st_size),
            "sha256": sha256_of_file(p),
        }
    )

validation_df = pd.DataFrame(validation_records)
print(f"Validated {len(required_files)} artifacts.")
validation_df.head(20)

Validated 25 artifacts.


,path,size_bytes,sha256
0,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,45598,6506f5bb792946ed8986d860fcdc914dac9d119a0cabf4...
1,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,83499974,7fb408841c0f71b8c2d7d30c1db5791176feddf9616c51...
2,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,19443463,56d1eef758a62cebdd9b072cb484dcab6283d4e7414a6f...
3,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,33823285,e37f7f28b571385168d12a284697e20f18a8d36c7f80e0...
4,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,83499974,7fb408841c0f71b8c2d7d30c1db5791176feddf9616c51...
5,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,6997135,392c87418681177ae7e8759b4d2456564c0d3487d61a27...
6,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,15975724,bce2f5d0974c6147c0fd2fafe388e7315dc63162862f06...
7,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,69726078,a5e491331b9ab4ee20b2134d9f9bb7e52414a315657338...
8,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,45503951,840ef980acce8eab435c179a8a1e79a6799b7451e1a8db...
9,C:\Users\yassi\Documents\PFE\Sciq Finetunning\...,66762315,ff92ec557a37dde1ffffb4945c8e6df795985f050661eb...
